# 06 · Hallucinations written by hand: does the curve notice a wrong fact that was not written hot?

**The question:** notebook 03 let Qwen hallucinate at a high temperature, so surprisal found every burst, but it
found the *hot writing*, true or not. Here the wrong words are written by hand into normal human texts, so nothing
was sampled hot: does the curve still react where a sentence becomes wrong, and does it react differently in a
factual text (a wrong fact) than in a story (a word that breaks the scene)?

**The texts.** 6 new real texts that Qwen had never read in this project (none shares an 8-word run with the 25
reference texts or the Qwen-written ones): the openings of three Wikipedia articles (Moon, Honey bee, Saturn) and a
passage from the middle of three Project Gutenberg books (Treasure Island, Peter Pan, The Wind in the Willows),
each ~690 words (sources in `manual_hallucinations/sources.csv`). Each has an **edited copy** with 7-9 words
swapped by hand so the sentence becomes wrong or absurd ("the sixth planet from the *hospital*"), and the
untouched **original**, so the two curves can be compared word by word.

**What is compared**, for the same three signals as notebooks 03-05 (GloVe top 5, LCA likely top 3 10%, surprisal
per word): the raw curve of the edited text against its original's, and the distance to real writing of both.
Then the whole-text checks of notebooks 04 and 05 (the 1-NN classifier learnt only from the 25 human texts, and
the rhythm). Qwen measures nouns and verbs only, so a swapped adjective ("hexagonal" -> "circular") can only show at
the next measured word.

Runtime: ~5 minutes for Qwen to read each of the 12 texts (GPU); finished steps are skipped.

In [1]:
import os
import sys
import time
import warnings
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")   # the model is already in the Hugging Face cache
warnings.filterwarnings("ignore")
# make this folder's qwen_config.py and analysis/ importable, wherever Jupyter was started (e.g. the repo root)
QWEN7B_DIR = Path.cwd()
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / "src/note_books/Qwen7B/qwen_config.py").exists():
        QWEN7B_DIR = folder / "src/note_books/Qwen7B"
        break
sys.path.insert(0, str(QWEN7B_DIR))

import analysis                                # also makes lca_core (repo root) importable
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from analysis.edits import word_edits
from qwen_config import (GUESSES_PER_WORD, LOAD_IN_4BIT, MANUAL_DIR, MANUAL_FIGURES_DIR, MANUAL_TABLES_DIR,
                         MANUAL_TOKENS_DIR, MANUAL_WORDS_DIR, MODEL_NAME, WHOLE_WORD_GUESSES)

for folder in [MANUAL_WORDS_DIR, MANUAL_TOKENS_DIR, MANUAL_FIGURES_DIR, MANUAL_TABLES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# each text: the original's name (e.g. factual_wiki_moon) and its edited copy's (factual_wiki_moon_manual)
original_names = []
for path in sorted((MANUAL_DIR / "original").glob("*.txt")):
    original_names.append(path.stem)
edited_names = []
for name in original_names:
    edited_names.append(f"{name}_manual")


def text_of(name):
    if name.endswith("_manual"):
        return (MANUAL_DIR / f"{name}.txt").read_text(encoding="utf-8")
    return (MANUAL_DIR / "original" / f"{name}.txt").read_text(encoding="utf-8")


model = None


def load_qwen():
    # the model is loaded only when something is left to read
    global model, tokenizer, meter
    if model is None:
        from lca_core import LCAMeter
        from src.model.load_model import load_model
        tokenizer, model = load_model(MODEL_NAME, load_in_4bit=LOAD_IN_4BIT)
        meter = LCAMeter(model=model, tokenizer=tokenizer)

## 1. The edits
Found by comparing each edited text with its original word by word (neighbouring changed words count as one edit).
Saved to `results/manual/tables/edits.csv`.

In [2]:
edit_rows = []
for name, edited_name in zip(original_names, edited_names):
    original, edited = text_of(name), text_of(edited_name)
    print(f"===== {edited_name}")
    for edit in word_edits(original, edited):
        place = edit["char edited"]
        print(f"  {edit['was']!r:>28} -> {edit['now']!r:<22} ...{edited[max(place - 50, 0):place + 40]}...")
        edit_rows.append({"text": edited_name, "original": name, **edit})
edits = pd.DataFrame(edit_rows)
edits.to_csv(MANUAL_TABLES_DIR / "edits.csv", index=False)
print(f"{len(edits)} edits in {edits['text'].nunique()} texts")

===== creative_gut_peter_pan_manual
                     'climbed' -> 'opened'               ...harp. First to draw blood was John, who gallantly opened into the boat and yelled Starkey....
                        'held' -> 'yelled'               ... was John, who gallantly opened into the boat and yelled Starkey. There was fierce struggl...
                          'He' -> 'Mike'                 ...ich the cutlass was torn from the pirate’s grasp. Mike wriggled overboard and John leapt a...
                        'him.' -> 'her.'                 ...asp. Mike wriggled overboard and John leapt after her. The dinghy drifted away. Here and t...
                        'claw' -> 'door'                 ...med for backing from the pirate captain. His iron door made a circle of dead water round h...
                    'breathe,' -> 'swim,'                ...the water that they met. Hook rose to the rock to swim, and at the same moment Peter scale...
                    'slippery' -> 'hot' 

## 2. Qwen reads every original and edited text
Same measurements as notebook 01. Texts with saved results are skipped.

In [3]:
for name in original_names + edited_names:
    words_path = MANUAL_WORDS_DIR / f"{name}.csv"
    tokens_path = MANUAL_TOKENS_DIR / f"{name}.csv"
    if words_path.exists() and tokens_path.exists():
        continue
    load_qwen()
    started = time.time()
    words = meter.run(text_of(name), k=GUESSES_PER_WORD, complete_words=WHOLE_WORD_GUESSES)
    words.to_csv(words_path, index=False)
    meter.last_tokens.to_csv(tokens_path, index=False)
    print(f"{name:42s} {len(words):4d} words  {time.time() - started:5.0f} s", flush=True)

system check:
Python path: C:\Users\yoav7\miniconda3\python.exe
Torch version: 2.6.0+cu124
GPU: NVIDIA GeForce RTX 4060
PyTorch: 2.6.0+cu124
CUDA: 12.4
CUDA available: True
Capability: (8, 9)
this is include the following functions torch , torch.nn.functional as F, transformers.AutoTokenizer, transformers.AutoModelForCausalLM


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

creative_gut_peter_pan                      188 words    353 s


creative_gut_treasure_island                215 words    322 s


creative_gut_wind_willows                   180 words    265 s


factual_wiki_honey_bee                      242 words    395 s


... (4 outputs left out)

creative_gut_wind_willows_manual            178 words    317 s


factual_wiki_honey_bee_manual               242 words    319 s


factual_wiki_moon_manual                    193 words    242 s


factual_wiki_saturn_manual                  192 words    248 s
